# 07 — NumPy Attention Implementation and Tests

    **Companion chapter:** `07-numpy-implementation.md`

    ## Learning goals

    - Implement numerically stable softmax.
- Validate attention input shapes.
- Reproduce and test the chapter's numerical result.
- Add causal masking and learned projections.
- Extend single-head attention to multiple heads.

    ## How to use this notebook

    Run the cells from top to bottom. Read the comments, change small values, and
    rerun the cell. Every notebook ends with practice prompts that can become
    GitHub issues, exercises, or discussion questions.

In [1]:
from __future__ import annotations

import math
import numpy as np

np.set_printoptions(precision=3, suppress=True)

## 1. A validated NumPy attention function

The function checks dimensions, supports an additive mask, and returns both
contextualized representations and attention probabilities.

In [2]:
def softmax(x: np.ndarray, axis: int = -1) -> np.ndarray:
    shifted = x - np.max(x, axis=axis, keepdims=True)
    exp_x = np.exp(shifted)
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)


def scaled_dot_product_attention(
    q: np.ndarray,
    k: np.ndarray,
    v: np.ndarray,
    mask: np.ndarray | None = None,
) -> tuple[np.ndarray, np.ndarray]:
    if q.ndim != 2 or k.ndim != 2 or v.ndim != 2:
        raise ValueError("q, k, and v must be two-dimensional arrays.")
    if q.shape[1] != k.shape[1]:
        raise ValueError("q and k must have the same feature dimension.")
    if k.shape[0] != v.shape[0]:
        raise ValueError("k and v must contain the same number of tokens.")

    scores = q @ k.T / np.sqrt(q.shape[-1])

    if mask is not None:
        if mask.shape != scores.shape:
            raise ValueError("mask must have the same shape as the score matrix.")
        scores = scores + mask

    weights = softmax(scores, axis=-1)
    output = weights @ v
    return output, weights

In [3]:
x = np.array(
    [
        [1.0, 0.0],
        [0.0, 1.0],
        [1.0, 1.0],
    ]
)

output, weights = scaled_dot_product_attention(x, x, x)

print("Weights:\n", weights)
print("\nOutput:\n", output)

Weights:
 [[0.401 0.198 0.401]
 [0.198 0.401 0.401]
 [0.248 0.248 0.503]]

Output:
 [[0.802 0.599]
 [0.599 0.802]
 [0.752 0.752]]


## 2. Unit-test the numerical example

Small assertions turn a teaching calculation into maintainable repository code.

In [4]:
expected_weights = np.array(
    [
        [0.401, 0.198, 0.401],
        [0.198, 0.401, 0.401],
        [0.248, 0.248, 0.503],
    ]
)

expected_output = np.array(
    [
        [0.802, 0.599],
        [0.599, 0.802],
        [0.752, 0.752],
    ]
)

assert np.allclose(weights, expected_weights, atol=1e-3)
assert np.allclose(output, expected_output, atol=1e-3)
assert np.allclose(weights.sum(axis=-1), 1.0)

print("All numerical checks passed.")

All numerical checks passed.


## 3. Add a causal mask

In [5]:
def make_causal_mask(length: int) -> np.ndarray:
    return np.triu(
        np.full((length, length), -np.inf),
        k=1,
    )


causal = make_causal_mask(len(x))
masked_output, masked_weights = scaled_dot_product_attention(
    x, x, x, mask=causal
)

print("Mask:\n", causal)
print("\nMasked weights:\n", masked_weights)
assert np.allclose(np.triu(masked_weights, k=1), 0.0)

Mask:
 [[  0. -inf -inf]
 [  0.   0. -inf]
 [  0.   0.   0.]]

Masked weights:
 [[1.    0.    0.   ]
 [0.33  0.67  0.   ]
 [0.248 0.248 0.503]]


## 4. Replace identity matrices with learned-style projections

In [6]:
w_q = np.array(
    [
        [1.0, 0.0],
        [0.5, 1.0],
    ]
)
w_k = np.array(
    [
        [0.5, 1.0],
        [1.0, 0.0],
    ]
)
w_v = np.array(
    [
        [1.0, 1.0],
        [0.0, 1.0],
    ]
)

q = x @ w_q
k = x @ w_k
v = x @ w_v

projected_output, projected_weights = scaled_dot_product_attention(q, k, v)

print("Q:\n", q)
print("\nK:\n", k)
print("\nV:\n", v)
print("\nWeights:\n", projected_weights)
print("\nOutput:\n", projected_output)

Q:
 [[1.  0. ]
 [0.5 1. ]
 [1.5 1. ]]

K:
 [[0.5 1. ]
 [1.  0. ]
 [1.5 1. ]]

V:
 [[1. 1.]
 [0. 1.]
 [1. 2.]]

Weights:
 [[0.225 0.32  0.456]
 [0.332 0.195 0.473]
 [0.212 0.177 0.611]]

Output:
 [[0.68  1.456]
 [0.805 1.473]
 [0.823 1.611]]


## 5. Multi-head attention in NumPy

This version expects one projection triplet per head.

In [7]:
def multi_head_attention(
    x: np.ndarray,
    w_q_heads: list[np.ndarray],
    w_k_heads: list[np.ndarray],
    w_v_heads: list[np.ndarray],
    w_o: np.ndarray,
) -> tuple[np.ndarray, list[np.ndarray]]:
    if not (len(w_q_heads) == len(w_k_heads) == len(w_v_heads)):
        raise ValueError("Each head needs Q, K, and V projection matrices.")

    head_outputs = []
    head_weights = []

    for w_q, w_k, w_v in zip(w_q_heads, w_k_heads, w_v_heads):
        q = x @ w_q
        k = x @ w_k
        v = x @ w_v
        output, weights = scaled_dot_product_attention(q, k, v)
        head_outputs.append(output)
        head_weights.append(weights)

    concatenated = np.concatenate(head_outputs, axis=-1)
    return concatenated @ w_o, head_weights


x4 = np.array(
    [
        [1.0, 0.0, 1.0, 0.0],
        [0.0, 1.0, 0.0, 1.0],
        [1.0, 1.0, 0.0, 0.0],
    ]
)

# Two heads, each maps d_model=4 to head_dim=2.
rng = np.random.default_rng(42)
w_q_heads = [rng.normal(size=(4, 2)) for _ in range(2)]
w_k_heads = [rng.normal(size=(4, 2)) for _ in range(2)]
w_v_heads = [rng.normal(size=(4, 2)) for _ in range(2)]
w_o = rng.normal(size=(4, 4))

mha_output, mha_weights = multi_head_attention(
    x4, w_q_heads, w_k_heads, w_v_heads, w_o
)

print("Multi-head output shape:", mha_output.shape)
print("Number of attention matrices:", len(mha_weights))
print("Each attention matrix:", mha_weights[0].shape)

Multi-head output shape: (3, 4)
Number of attention matrices: 2
Each attention matrix: (3, 3)


## 6. Test failure cases

In [8]:
try:
    scaled_dot_product_attention(
        np.ones((3, 2)),
        np.ones((3, 4)),
        np.ones((3, 2)),
    )
except ValueError as error:
    print("Expected error:", error)

try:
    scaled_dot_product_attention(
        np.ones((3, 2)),
        np.ones((3, 2)),
        np.ones((4, 2)),
    )
except ValueError as error:
    print("Expected error:", error)

Expected error: q and k must have the same feature dimension.
Expected error: k and v must contain the same number of tokens.


## Practice

1. Add a dropout-like mask to the attention probabilities.
2. Extend the function to accept batched 3-D arrays.
3. Add a key-padding mask and combine it with the causal mask.
4. Write `pytest` tests in a separate `tests/` directory.
5. Compare this NumPy output with `torch.nn.functional.scaled_dot_product_attention`.